# Inspect any product — plot all bands

Point `PRODUCT` at any `.zarr` or `.zarr.zip` image dataset and plot every band as its own figure.
Works for our canonical L0 (`band{NN}`), real/public L0 and L1B (`img`), and L1A (`l1a_raw_image`).

- Store root: `$S2_DATA_STORE` (default `~/data-store`) — same convention as `scripts/run_pipeline.py`.
- Needs only `numpy + zarr + matplotlib` (the plain generator env; no eopf).
- Set `DETECTOR = 1..12` to focus on one detector; leave `None` to plot every detector × band.

In [14]:
# One-time kernel setup — safe to re-run. After installing, RESTART the kernel.
import sys

_need_pkg = _need_deps = False
try:
    import s2_msi_raw_generator  # noqa: F401
except Exception:
    _need_pkg = True
try:
    import matplotlib.pyplot  # noqa: F401
    import zarr  # noqa: F401
except Exception:
    _need_deps = True
if _need_pkg:
    !{sys.executable} -m pip install --quiet -e .. --no-deps
if _need_deps:
    !{sys.executable} -m pip install --quiet matplotlib zarr
    !{sys.executable} -m pip install --quiet --force-reinstall --no-deps matplotlib zarr
if _need_pkg or _need_deps:
    print("installed/repaired — RESTART the kernel (Kernel ▸ Restart Kernel…) and run all cells again")
else:
    print("environment OK")

environment OK


In [15]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from s2_msi_raw_generator.io import _open

plt.rcParams["figure.dpi"] = 75

STORE = Path(os.environ.get("S2_DATA_STORE") or "~/data-store").expanduser()
PRODUCT = os.environ.get("S2_INSPECT_PRODUCT", "")  # full/relative path; leave blank to use index
PRODUCT_INDEX = 8  # list index when PRODUCT is blank
DETECTOR = None  # None = every detector; or 1..12
MAX_LINES = 2048  # along-track window per band


def du_mb(p: Path) -> float:
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6 if p.exists() else 0.0


print(f"store: {STORE}\n")
print("directories:")
for sub in ("inputs", "caldb", "l0", "l1a_prime", "l1b", "quicklook", "figures", "report"):
    d = STORE / sub
    if d.is_dir():
        n = sum(1 for _ in d.iterdir())
        print(f"  {sub:12s}  {d}  {n:4d} entries  {du_mb(d):9.1f} MB")
    else:
        print(f"  {sub:12s}  {d}  (missing)")

found = []
_seen = set()
for p in sorted(STORE.rglob("*.zarr")):
    if not p.is_dir():
        continue
    key = p.resolve()
    if key in _seen:
        continue
    _seen.add(key)
    found.append(str(p))
for p in sorted(STORE.rglob("*.zarr.zip")):
    key = p.resolve()
    if key in _seen:
        continue
    _seen.add(key)
    found.append(str(p))
print(f"\n{len(found)} products:")
for i, p in enumerate(found):
    rel = Path(p).relative_to(STORE).as_posix() if Path(p).is_relative_to(STORE) else Path(p).name
    print(f"  [{i:2d}] {rel}")

store: /home/jovyan/data-store

directories:
  inputs        /home/jovyan/data-store/inputs     5 entries     9779.7 MB
  caldb         /home/jovyan/data-store/caldb     8 entries        3.8 MB
  l0            /home/jovyan/data-store/l0     2 entries      761.2 MB
  l1a_prime     /home/jovyan/data-store/l1a_prime     3 entries      191.1 MB
  l1b           /home/jovyan/data-store/l1b     0 entries        0.0 MB
  quicklook     /home/jovyan/data-store/quicklook     7 entries       24.9 MB
  figures       /home/jovyan/data-store/figures     0 entries        0.0 MB
  report        /home/jovyan/data-store/report    12 entries        0.4 MB

25 products:
  [ 0] caldb/S02MSIDCA_20240403T102415_0001_A045_T137.zarr
  [ 1] caldb/S02MSISCA_20240403T102415_0001_A045_T536.zarr
  [ 2] caldb/dark.zarr
  [ 3] caldb/noise.zarr
  [ 4] caldb/nuc.zarr
  [ 5] caldb/radiometric.zarr
  [ 6] caldb/spectral.zarr
  [ 7] inputs/PDI_MSI_S2_L1A.zarr
  [ 8] inputs/s2-sensor/PDI_MSI_S2_L1A/PDI_MSI_S2_L1A.zarr
  [ 9

Set `PRODUCT_INDEX` to a list index, or set `PRODUCT` to a full/relative path from the list, then run the next cell.

In [ ]:
def _pick_image_array(arrays):
    """One display array per band group — prefer l1a_raw_image > band* > img."""
    if "l1a_raw_image" in arrays:
        return "l1a_raw_image", arrays["l1a_raw_image"]
    for name in sorted(arrays):
        if name.startswith("band"):
            return name, arrays[name]
    if "img" in arrays:
        return "img", arrays["img"]
    return None


def iter_bands(group, prefix=""):
    imgs = {
        name: arr
        for name, arr in group.arrays()
        if getattr(arr, "ndim", 0) == 2
        and (name.startswith("band") or name in ("img", "l1a_raw_image"))
    }
    picked = _pick_image_array(imgs)
    if picked:
        name, arr = picked
        yield f"{prefix}{name}", arr
    for name, sub in sorted(group.groups(), key=lambda kv: kv[0]):
        yield from iter_bands(sub, f"{prefix}{name}/")


def show(arr, title, max_lines=MAX_LINES):
    img = np.asarray(arr[:max_lines])
    lo, hi = np.percentile(img, (2, 98))
    plt.figure(figsize=(9, 5))
    plt.imshow(
        np.clip(img, lo, max(hi, lo + 1e-9)),
        cmap="gray",
        aspect="auto",
        interpolation="nearest",
    )
    plt.title(f"{title}  [{lo:.0f}-{hi:.0f} DN]  {img.shape}")
    plt.xlabel("column")
    plt.ylabel("line")
    plt.colorbar(label="DN")
    plt.tight_layout()
    plt.show()


def resolve_product(product, index, found, store):
    if product:
        p = Path(product).expanduser()
        if p.exists():
            return str(p)
        rel = store / product
        if rel.exists():
            return str(rel)
        raise FileNotFoundError(f"PRODUCT not found: {product}")
    if not found:
        return ""
    if not (0 <= index < len(found)):
        raise IndexError(f"PRODUCT_INDEX {index} out of range (0..{len(found) - 1})")
    return found[index]


target = resolve_product(PRODUCT, PRODUCT_INDEX, found, STORE)
if not target:
    print(f"no products under {STORE} — run: python scripts/run_pipeline.py")
else:
    g = _open(str(target))
    bands = list(iter_bands(g))
    if DETECTOR is not None:
        tag = (f"d{DETECTOR:02d}/", f"DD{DETECTOR:02d}/")
        bands = [(pth, a) for pth, a in bands if any(t in pth for t in tag)]
    print(f"{Path(target).name}: plotting {len(bands)} band(s)\n")
    for pth, arr in bands:
        show(arr, f"{Path(target).name} — {pth}")
    if not bands:
        print("no image bands found (annotation-only product?) — try another PRODUCT")

import json

from s2_msi_raw_generator import inventory, metadata

pdi_candidates = sorted(STORE.rglob("PDI_MSI_S2_L1A.zarr"))
print("\n" + "=" * 60)
if not pdi_candidates:
    print("PDI_MSI_S2_L1A.zarr not found under store")
else:
    pdi_path = pdi_candidates[0]
    print(f"PDI_MSI_S2_L1A metadata — {pdi_path.relative_to(STORE)}")
    print("=" * 60)
    pdi_g = _open(str(pdi_path))
    attrs = dict(pdi_g.attrs)
    print(f"\nroot attrs ({len(attrs)} keys):", sorted(attrs.keys()) or "(empty)")
    if attrs:
        print(json.dumps(attrs, indent=2, default=str))
    _, props, stac_flags = metadata.normalise_stac(attrs)
    print("\nSTAC properties:")
    if props:
        for k in sorted(props):
            print(f"  {k}: {props[k]}")
    else:
        print("  (none)")
    if stac_flags:
        print("\nSTAC flags:", ", ".join(stac_flags))
    ident = inventory.read_zarr_identity(pdi_path)
    print("\nacquisition identity:")
    print(json.dumps(ident, indent=2, default=str))

In [59]:
import re
from datetime import datetime
from pathlib import Path

from s2_msi_raw_generator.io import _open

dir_path = Path("/home/jovyan/data-store/inputs/public-data")
needle = "20240403"

products = sorted(p for p in dir_path.rglob(f"*{needle}*.zarr") if p.is_dir())
products += sorted(dir_path.rglob(f"*{needle}*.zarr.zip"))

# Product name pattern:  S02MSI<LVL>_<YYYYMMDDgTHHMMSS>_<NNNN>_A<orbit>_T<tile>
NAME_RE = re.compile(
    r"^(?P<mission>S02MSI\w+?)_(?P<dt>\d{8}T\d{6})_(?P<counter>\d+)"
    r"_A(?P<orbit>\d+)_T(?P<tile>\w+)"
)


def find_key(obj, key):
    """Find a key anywhere in the attrs tree (returns first match)."""
    if isinstance(obj, dict):
        if key in obj and not isinstance(obj[key], (dict, list)):
            return obj[key]
        for v in obj.values():
            r = find_key(v, key)
            if r is not None:
                return r
    elif isinstance(obj, list):
        for v in obj:
            r = find_key(v, key)
            if r is not None:
                return r
    return None


for product in products:
    g = _open(str(product))
    attrs = dict(g.attrs)
    stac = attrs.get("stac_discovery", {}) or {}
    props = stac.get("properties", {}) or {}

    # 1) STAC properties first, 2) search attrs tree, 3) finally filename
    m = NAME_RE.match(product.name)
    dt = props.get("start_datetime") or props.get("datetime") \
        or find_key(attrs, "start_datetime") or find_key(attrs, "datetime")
    if dt is None and m:
        dt = datetime.strptime(m["dt"], "%Y%m%dT%H%M%S").isoformat()
        dt_source = "filename"
    else:
        dt_source = "metadata"

    platform = props.get("platform") or find_key(attrs, "platform") or stac.get("id")
    orbit = props.get("sat:relative_orbit") or find_key(attrs, "sat:relative_orbit")
    if orbit is None and m:
        orbit = f"A{m['orbit']} (from filename)"

    print(product.name)
    print(f"  Acquisition date : {dt}   [{dt_source}]")
    print(f"  Platform     : {platform}")
    print(f"  Orbit        : {orbit}")
    if m:
        print(f"  Level/Tile   : {m['mission']} / T{m['tile']}")
    print("-" * 80)


S02MSIL0P_20240403T000000_0001_A123_T000.zarr
  Acquisition date : 2024-04-03T00:00:00   [filename]
  Platform     : S2MSIL0plus
  Orbit        : A123 (from filename)
  Seviye/Tile  : S02MSIL0P / T000
--------------------------------------------------------------------------------
S02MSIL1A_20240403T000000_0001_A123_T000.zarr
  Acquisition date : 2024-04-03T00:00:00   [filename]
  Platform     : S2MSIL1A
  Orbit        : A123 (from filename)
  Seviye/Tile  : S02MSIL1A / T000
--------------------------------------------------------------------------------
S02MSIL1B_20240403T000000_0001_A123_T000.zarr
  Acquisition date : 2024-04-03T00:00:00   [filename]
  Platform     : S2MSIL1B
  Orbit        : A123 (from filename)
  Seviye/Tile  : S02MSIL1B / T000
--------------------------------------------------------------------------------
S02MSIL0P_20240403T000000_0001_A123_T000.zarr.zip
  Acquisition date : 2024-04-03T00:00:00   [filename]
  Platform     : S2MSIL0plus
  Orbit        : A123 (from

In [61]:
# Sensor characterization + PSF data: WHEN produced, WHAT is inside?
from datetime import datetime, timezone
from pathlib import Path

from s2_msi_raw_generator.io import _open

roots = [
    Path("/home/jovyan/validation-data/data-store/inputs/s2-sensor"),
    Path("/home/jovyan/validation-data/data-store/inputs/psf"),
]

DATE_HINTS = ("creation", "created", "date", "datetime", "valid", "validity",
              "generation", "processing", "start", "stop", "end", "version", "provenance")


def find_dateish(obj, path="", out=None):
    out = {} if out is None else out
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f"{path}.{k}" if path else k
            if isinstance(v, (dict, list)):
                find_dateish(v, p, out)
            elif any(h in k.lower() for h in DATE_HINTS):
                out[p] = v
    elif isinstance(obj, list):
        for i, v in enumerate(obj[:3]):
            find_dateish(v, f"{path}[{i}]", out)
    return out


for root in roots:
    print("#" * 90)
    print("# ROOT:", root, "" if root.exists() else "  <-- MISSING / inaccessible")
    print("#" * 90)
    if not root.exists():
        continue

    entries = sorted(root.iterdir())
    print(f"\n{len(entries)} entries (fs timestamps — when data was written to disk):")
    for e in entries:
        st = e.stat()
        mtime = datetime.fromtimestamp(st.st_mtime, timezone.utc)
        ctime = datetime.fromtimestamp(st.st_ctime, timezone.utc)
        kind = "dir " if e.is_dir() else "file"
        print(f"  {kind}  mtime={mtime:%Y-%m-%d %H:%M}  ctime={ctime:%Y-%m-%d %H:%M}  {e.name}")

    zarrs = sorted(list(root.rglob("*.zarr")) + list(root.rglob("*.zarr.zip")))
    zarrs = [z for z in zarrs if z.suffix == ".zip" or z.is_dir()]
    for z in zarrs[:8]:
        try:
            attrs = dict(_open(str(z)).attrs)
        except Exception as exc:
            print(f"\n  [{z.relative_to(root)}] could not open: {exc}")
            continue
        dates = find_dateish(attrs)
        print(f"\n  --- {z.relative_to(root)} ---")
        print(f"      attrs keys: {sorted(attrs.keys())[:12]}")
        if dates:
            print("      date/provenance fields:")
            for k, v in dates.items():
                print(f"        {k} = {v}")
        else:
            print("      (no date/provenance field in metadata)")
    print()


##########################################################################################
# ROOT: /home/jovyan/validation-data/data-store/inputs/s2-sensor 
##########################################################################################

4 entries (fs timestamps — when data was written to disk):
  dir   mtime=2026-07-03 23:09  ctime=2026-07-03 23:09  GIPP
  file  mtime=2026-07-03 23:09  ctime=2026-07-03 23:09  GIPP.tgz
  dir   mtime=2026-07-03 23:08  ctime=2026-07-03 23:08  PDI_MSI_S2_L1A
  dir   mtime=2026-07-03 23:09  ctime=2026-07-03 23:09  RADIO_AB_input

  --- PDI_MSI_S2_L1A/PDI_MSI_S2_L1A.zarr ---
      attrs keys: []
      (no date/provenance field in metadata)

  --- RADIO_AB_input/PDI_MSI_S2_L1A.zarr ---
      attrs keys: []
      (no date/provenance field in metadata)

##########################################################################################
# ROOT: /home/jovyan/validation-data/data-store/inputs/psf 
################################################

In [62]:
# Real provenance/production info: PROVENANCE.md + GIPP filenames + zarr low-level attrs
import re
from pathlib import Path

from s2_msi_raw_generator.io import _open

VROOT = Path("/home/jovyan/validation-data/data-store/inputs")

# 1) PROVENANCE.md and all text/sidecar files (provenance written here)
print("=" * 90)
print("1) PROVENANCE / text files")
print("=" * 90)
for txt in sorted(VROOT.rglob("*.md")) + sorted(VROOT.rglob("*.json")) + sorted(VROOT.rglob("*.xml")):
    if txt.is_file() and txt.stat().st_size < 200_000:
        print(f"\n----- {txt.relative_to(VROOT)} -----")
        print(txt.read_text(errors="replace")[:3000])

# 2) GIPP filenames — date + validity range encoded in name per S2 convention
print("\n" + "=" * 90)
print("2) GIPP files (names encode date/validity)")
print("=" * 90)
gipp = VROOT / "s2-sensor" / "GIPP"
if gipp.exists():
    for f in sorted(gipp.rglob("*")):
        if f.is_file():
            print(f"  {f.relative_to(gipp)}")
            # S2 name: ..._YYYYMMDDTHHMMSS_..._VYYYYMMDDTHHMMSS_YYYYMMDDTHHMMSS
            for tag in re.findall(r"\d{8}T\d{6}", f.name):
                print(f"      -> date token: {tag}")

# 3) search attrs at EVERY level in zarr (may be in subgroups even if root empty)
print("\n" + "=" * 90)
print("3) low-level attrs in sensor zarrs")
print("=" * 90)


def walk_attrs(node, prefix="/"):
    a = dict(getattr(node, "attrs", {}))
    if a:
        print(f"  {prefix}  attrs: {list(a.keys())}")
        for k, v in a.items():
            if not isinstance(v, (dict, list)):
                print(f"      {k} = {v}")
    for name, sub in getattr(node, "groups", lambda: [])():
        walk_attrs(sub, prefix + name + "/")
    for name, arr in getattr(node, "arrays", lambda: [])():
        aa = dict(getattr(arr, "attrs", {}))
        if aa:
            print(f"  {prefix}{name}  (array) attrs: {list(aa.keys())}")
            for k, v in aa.items():
                if not isinstance(v, (dict, list)):
                    print(f"      {k} = {v}")


for z in sorted((VROOT / "s2-sensor").rglob("*.zarr")):
    if z.is_dir():
        print(f"\n--- {z.relative_to(VROOT)} ---")
        walk_attrs(_open(str(z)))


1) PROVENANCE / text files

----- psf/PROVENANCE.md -----
# Real Sentinel-2 MSI Point Spread Functions

**Source:** SentiWiki (Copernicus) — <https://sentiwiki.copernicus.eu/web/s2-documents>
(`S2A_PSF.zip`, `S2B_PSF.zip`, `S2C_PSF.zip`, modified 2025-11-07). Copernicus open data.

These are the **official ESA per-band PSF matrices** used by the E2ES S6 re-blur step — they
replace the earlier synthetic Gaussian-from-MTF kernels.

**Format (per ESA notice):**
- One CSV per band per unit: `S2{A,B,C}_PSF_B{NN}.csv`.
- Each file is a **33×33** matrix, **normalised** (Σ = 1), **oversampling factor 5**, centre pixel
  at matrix coordinate (17, 17) (1-indexed).
- Derived from MTF measurements at Nyquist (along-track + across-track), Gaussian-modelled:
  S2A & S2C from 2024 acquisitions, S2B from 2023.
- PSFs correspond to **L1B products** (focal-plane geometry, after binning), all bands **except
  B10** (water-vapour band — does not see the ground).

At load time `s2_msi_raw_generator.adf` in